# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malaikasaleem944/malaika_flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

For this lane, one row represents one content item for one client on one report date.

The main analysis window for this task is the mid-panel month of March 2026 (`month=2026-03`). The warehouse contains daily performance history from 2025-01-27 through 2026-06-30, but I will use March 2026 for feature development and verification rather than the final June 2026 sample.

The decision is made at the content-item level: which content should an editor prioritize for review or refresh?

In [ ]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

## 2. Fields: feature / label / context / excluded

### Features
These are fields that are knowable at the decision moment and can be used to rank content:

- `impressions_90d` — recent historical visibility available before the decision.
- `clicks_90d` — historical search clicks available before the decision.
- `sessions_90d` — historical sessions available before the decision.
- `gsc_avg_position` — historical search position available before the decision.
- `content_age_days` — content age is known at the decision moment.
- `days_since_last_update` — freshness information is known before the decision.
- `search_volume` — search demand information available before the decision.
- `competition` — competition information available before the decision.

I will keep the feature set to five features in the final feature frame, as required by the assignment.

### Label / proxy
The label is the observed future content-performance outcome used to evaluate whether the ranking identified content that needed attention.

A future decline outcome should be measured after the feature window. It must not be calculated from information available at the decision moment.

### Context
These fields help identify, group, join, or interpret observations but should not be model features:

- `content_id` — identifies the content item.
- `client_id` — identifies the client and can be used for grouping or splitting.
- `report_date` — identifies the observation date and defines the time window.

### Excluded
- `trend_direction` — excluded because it is an outcome-derived field and would leak the target into the features.
- `trend_pct` — excluded because it is used to derive the decline outcome and is therefore future/outcome information for the prediction task.
- Final-month (`2026-06`) data — excluded from development because it is the sealed final month and may represent the natural outcome window.
- Any fields that are only available after the prediction/decision moment — excluded to prevent future-information leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)


I will verify the warehouse grain, the March 2026 row count and date span, and data availability using three queries. I will use the mid-panel March 2026 partition for development and keep the final June 2026 month sealed.

In [ ]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

**Grain result:** The query returned zero duplicate `(report_date, client_id, content_id)` combinations. This verifies that one row represents one content item for one client on one report date in the March 2026 partition.

In [ ]:
count_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

count_window

In [ ]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS available_rows
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

availability_check

In [ ]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

## 4. Data limits

This data cannot provide a perfectly balanced history for every client and content item because clients have different amounts of available search and analytics history.

Some early rows may contain Google Analytics fields that are zero-filled when `ga4_data_available` is not true. Therefore, a zero value does not always mean that there was no engagement.

The fixed 90-day query window can also overlap with the performance snapshot window. Therefore, features and labels must be aligned carefully so that information from the future outcome window does not enter the features.

The final June 2026 month is treated as a sealed outcome/test period rather than being used to develop the label or features.

Therefore, the model can support content-prioritization decisions, but it cannot prove that a content change caused a future performance improvement.

## Self-check

- [ ] Every section above is filled with both the markdown explanation and the code that supports it.
- [ ] The notebook runs from top to bottom with no errors after using Runtime → Run all.
- [ ] No client names, private URLs, access tokens, or private queries are included.
- [ ] My claims use careful language such as observed, measured, directional, or decision-support.
- [ ] The notebook is committed under `work/notebooks/` and pushed to my repository.